# MARV × Titans — causal ablation: does one neuron store one fact? (Colab T4 / CPU)

`marv_titans_memdiff_colab.ipynb` found that a **trained** Titans memory shows far fewer
write-collisions than an untrained one, but a sanity check there showed that drop was driven
by aggressive **forgetting** (the trained memory holds almost nothing by end-of-document), not
by clean per-fact storage. Whether storage actually *localizes* to individual hidden units at
the moment of writing was left **unanswered** — the only tool available was a correlational
proxy (align a unit's weight-delta against the chunk-mean of random vectors), which is
near-degenerate.

This notebook answers it with **causation instead of correlation**: store distinct, tracked
key→value pairs, then actually zero out one hidden unit at a time and re-measure recall of
every pair. This is the same move as MARV's core workflow on a static model —
`constellation` (find candidate units) → `suppress`/`ablate` (turn them off) → measure what
broke — just re-implemented for a memory that's being written *live*, during inference,
instead of one baked in at training time.

In [ ]:
!pip install -q titans-pytorch matplotlib
!git clone -q -b marv-titan https://github.com/thebnbrkr/marv.git /content/marv
import sys; sys.path.insert(0, '/content/marv/experiments')

import numpy as np, torch, matplotlib.pyplot as plt
from titans_pytorch import NeuralMemory
from titans_memdiff import train_recall
from titans_ablation import (
    DIM, HIDDEN, BREAK_THRESHOLD,
    store_tracked_pairs, ablate, recall_cosines, check_replay_fidelity, run_ablation_sweep,
    per_pair_sensitivity, per_unit_concentration,
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
N_PAIRS = 12
torch.manual_seed(0); np.random.seed(0)
print('device:', device, ' memory MLP:', f'{DIM}->{HIDDEN}->{DIM}', ' tracked pairs:', N_PAIRS)

## 0. Sanity check first — does our retrieval call even reproduce the real model?

The earlier prototype's ablation attempt called `functional_call(mem.memory_model, weights, q)`
directly — skipping the pre/post-processing `NeuralMemory.retrieve_memories()` normally does
around that call (a pre-norm, multi-head split, a query norm, a multihead RMSNorm, an optional
retrieve gate, then a head merge). That mismatch is almost certainly why it only matched true
retrieval at cos ≈ 0.6. Here we call the library's own public `retrieve_memories()` method
instead. If this cell doesn't print ~1.0, stop — nothing below can be trusted.

In [ ]:
seq = torch.randn(1, 20, DIM, device=device)
mem_check = NeuralMemory(dim=DIM, chunk_size=1).to(device)
fidelity = check_replay_fidelity(mem_check, seq)
print(f'replay-fidelity cos (want ~1.0): {fidelity:.4f}')

## 1. Build an untrained and a trained memory

Same autoassociative recall task as the diff notebook: query with a random vector, the memory
should return that same vector. This teaches the *slow/outer* weights how to write and how
aggressively to forget — not any specific fact.

In [ ]:
mem_raw = NeuralMemory(dim=DIM, chunk_size=1).to(device)

mem_trained = NeuralMemory(dim=DIM, chunk_size=1).to(device)
print('training the memory on autoassociative recall (this is the slow part on CPU)...')
train_recall(mem_trained, steps=400, device=device)

## 2. Store N tracked pairs, measure baseline recall

Each pair is its own key **and** value (same convention as training). Store them one at a
time, then query each key and see how well the memory returns its matching value — *before*
touching anything. Pairs are in store order, so the last bars are the most recently written.

In [ ]:
pairs_raw, weights_raw = store_tracked_pairs(mem_raw, N_PAIRS, DIM, device, seed=0)
pairs_trained, weights_trained = store_tracked_pairs(mem_trained, N_PAIRS, DIM, device, seed=0)

baseline_raw = recall_cosines(mem_raw, weights_raw, pairs_raw)
baseline_trained = recall_cosines(mem_trained, weights_trained, pairs_trained)

fig, ax = plt.subplots(figsize=(7, 3.5))
x = np.arange(N_PAIRS)
ax.bar(x - 0.2, baseline_raw, width=0.4, label='untrained')
ax.bar(x + 0.2, baseline_trained, width=0.4, label='trained')
ax.set_xlabel('pair (store order -> right = most recent)'); ax.set_ylabel('recall cos sim')
ax.set_title('Baseline recall per pair'); ax.legend(); ax.axhline(0, color='grey', lw=0.5)
plt.tight_layout(); plt.show()

print('untrained baseline:', np.round(baseline_raw, 3))
print('trained baseline:  ', np.round(baseline_trained, 3))

**Expected pattern (2026-09-11 run):** untrained recall hovers near 0 everywhere — there's no
real recall function yet. Trained recall is much better overall, and strongly **recency-biased**
— the last few pairs stored recall far better than the earliest ones. This is the forgetting
curve from the diff notebook, now shown through direct recall instead of weight-snapshot norms —
two independent measurements agreeing is a good sign the effect is real.

## 3. The real test — ablate one unit at a time, watch what breaks

For every one of the 256 hidden units: zero its input column and output row, re-run recall on
every pair, and record the drop from baseline. A unit that cleanly breaks **one** pair and
leaves the rest alone is a real, causally-verified localized store. A unit that breaks **several**
pairs at once is a real, causally-verified collision. This can take a minute or two on CPU
(256 units × 2 memories).

In [ ]:
baseline_raw2, drop_raw = run_ablation_sweep(mem_raw, weights_raw, pairs_raw)
baseline_trained2, drop_trained = run_ablation_sweep(mem_trained, weights_trained, pairs_trained)

for label, drop in [('untrained', drop_raw), ('trained', drop_trained)]:
    hit = np.abs(drop) > BREAK_THRESHOLD
    hit_counts = hit.sum(axis=1)
    print(f'{label:>10}: largest single-unit effect on any pair = {np.abs(drop).max():.3f}  '
          f'(threshold {BREAK_THRESHOLD})')
    print(f'{"":>10}  units breaking 0 / 1 / >1 pairs hard: '
          f'{(hit_counts==0).sum()} / {(hit_counts==1).sum()} / {(hit_counts>1).sum()}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, label, drop in zip(axes, ['untrained', 'trained'], [drop_raw, drop_trained]):
    im = ax.imshow(np.abs(drop).T, aspect='auto', cmap='viridis', vmin=0, vmax=0.2)
    ax.set_title(f'{label}: |recall drop| per (unit, pair)')
    ax.set_xlabel('hidden unit'); ax.set_ylabel('pair')
fig.colorbar(im, ax=axes, shrink=0.8, label='|drop| (cos)')
plt.show()

print('If storage were localized, this heatmap would show a few bright, isolated cells (one')
print('unit x one pair). If it is diffuse, effects stay small and spread across many cells.')

**Expected pattern:** no cell lights up anywhere near the break threshold — the whole heatmap
stays a faint, spread-out haze. That is the causal, ablation-verified version of the earlier
correlational "the write is diffuse" finding: **no single hidden unit is a dedicated storage
slot for any one fact**, trained or not. Every unit nudges several pairs a little; none holds
one pair a lot.

## 4. A clearer read than the heatmap — collapse it two ways

The raw (unit, pair) heatmap above is honest but genuinely hard to read at a glance — real
structure and noise both look like faint speckle. Two collapsed views make the same data much
easier to judge:

- **Per-pair fragility** — average the effect across all 256 units for each pair, with a
  z-score against the other pairs. This directly answers "is this particular fact more
  easily disturbed than the others," instead of asking you to spot a faint horizontal band.
- **Per-unit concentration curve** — sort units by total causal effect and plot the running
  share of total importance. A steep early climb means a few units matter a lot (localized);
  a straight diagonal-ish line means every unit contributes about equally (diffuse) — the same
  Gini-style read `titans_memdiff.py` already uses for write concentration, applied here to
  *causal* importance instead of raw write magnitude.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, label, drop in zip(axes, ['untrained', 'trained'], [drop_raw, drop_trained]):
    mean, std, z = per_pair_sensitivity(drop)
    colors = ['crimson' if zi > 1.5 else ('steelblue' if zi < -1.5 else 'lightgrey') for zi in z]
    ax.bar(np.arange(len(mean)), mean, yerr=std, color=colors, capsize=3)
    ax.set_title(f'{label}: per-pair fragility (mean |drop| ± std across units)')
    ax.set_xlabel('pair (store order -> right = most recent)')
    ax.set_ylabel('mean |recall drop| across all 256 units')
plt.tight_layout(); plt.show()
print('red = notably MORE fragile than the pack (z>1.5), blue = notably more robust (z<-1.5),')
print('grey = unremarkable. See the seed-stability cell below before trusting a red bar from')
print('a single run.')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for label, drop in [('untrained', drop_raw), ('trained', drop_trained)]:
    _, cum_share = per_unit_concentration(drop)
    ax.plot(np.arange(1, HIDDEN + 1) / HIDDEN, cum_share, label=label)
ax.plot([0, 1], [0, 1], '--', color='grey', lw=1, label='perfectly uniform')
ax.set_xlabel('fraction of units, ranked by importance'); ax.set_ylabel('cumulative share of total causal effect')
ax.set_title('Is causal importance concentrated in a few units, or spread evenly?')
ax.legend(); plt.tight_layout(); plt.show()
print('A curve close to the dashed diagonal = diffuse (no small set of units dominates).')
print('A curve that bows sharply above the diagonal early = a few units carry most of the effect.')

## 5. Is a fragile-looking pair real, or just this run's noise?

A single run's z-scores can't tell the difference between "this pair is structurally more
fragile" and "this particular random draw happened to land that way." The real test: repeat
the whole pipeline across several independent seeds and check whether the **same store-position**
keeps coming up as most fragile, or whether it moves around each time.

**2026-09-11 result:** across 5 untrained seeds, the top-fragile position was 8, 4, 3, 4, 2 —
different every time except a 2/5 coincidence on position 4. That's consistent with pure noise,
not a structural effect — expected, since an untrained memory has no real recall function to
begin with. The trained-memory version of this check (below) is the one that actually tests
whether a "middle pairs are more fragile" effect is real. It's slow (~15-20 min for 5 seeds on
CPU) — run it if you want the honest answer rather than trusting one heatmap.

In [ ]:
# Slow (~15-20 min on CPU for 5 trained seeds) -- uncomment to run.
# from titans_ablation import seed_stability_check
# seed_stability_check(train=True, steps=200, n_pairs=N_PAIRS, seeds=(0, 1, 2, 3, 4))

## What this shows, and what's still open

**Answered (causally, not by proxy):** a trained Titans `NeuralMemory` here does **not**
localize a stored fact to one hidden unit. Ablating any single unit moves recall by at most a
few hundredths of cosine similarity, spread thinly across several pairs — never concentrated.
Combined with the forgetting-curve result, the picture is: this memory's storage is genuinely
**distributed / holographic**, and on top of that, aggressively forgetful. Not "a few tidy
drawers that empty fast" — more like no drawers at all, just a haze that fades.

**Still open:**
1. **Group ablation.** Does removing the *top-k* units most implicated in one pair (by summed
   |drop|) break that pair, even though no single one of them does alone? That would tell us
   whether the fact lives in a small coalition rather than either "one neuron" or "uniformly
   everywhere."
2. **A middle-pairs-are-more-fragile hypothesis, currently unconfirmed.** Section 4's per-pair
   view can surface a pair that looks more fragile than its neighbours. Section 5's seed-stability
   check is what separates a real positional effect from one run's noise -- run it before citing
   a fragile-looking pair as a finding.
3. **Scale.** Bigger `dim`, more memory layers, more tracked pairs — does distributed storage
   hold, or does a localization regime appear at a different scale?
4. **Real vocabulary.** This is still random vectors, not facts. Wiring the memory into a real
   tiny LM (`titans_pytorch.MemoryAsContextTransformer` on real text) would let a logit-lens
   read *what* a unit promotes, not just *whether* ablating it moves a number.

See `experiments/README.md` and `experiments/titans_ablation.py` on the `marv-titan` branch.